# Lung Cancer Detection Workflow

This notebook turns the whole `src` solution into one teaching-style TensorFlow/Keras workflow.


In [ ]:
from pathlib import Path
import json
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import pydicom
import tensorflow as tf
from tensorflow import keras
from tqdm import tqdm


## Annotation Helpers

These functions read the XML annotation files.


In [ ]:
def safe_text(element, tag):
    if element is None:
        return None
    node = element.find(tag)
    if node is None or node.text is None:
        return None
    value = node.text.strip()
    return value or None

def normalize_patient_id(text):
    if not text:
        return None
    text = text.strip()
    if text.lower().startswith('lung_dx-'):
        return text.split('-', 1)[1]
    return text

def build_dicom_index(image_root):
    image_root = Path(image_root)
    dicom_files = sorted(image_root.rglob('*.dcm'))
    if not dicom_files:
        raise FileNotFoundError(f'No DICOM files found under {image_root}')
    index = {'by_uid': {}, 'by_patient': {}, 'by_name': {}}
    for dcm_path in tqdm(dicom_files, desc='Indexing DICOM files'):
        try:
            relative = dcm_path.relative_to(image_root)
            patient_folder = relative.parts[0] if relative.parts else dcm_path.parent.name
        except ValueError:
            patient_folder = dcm_path.parent.name
        patient_id = normalize_patient_id(patient_folder)
        index['by_patient'].setdefault(patient_id, []).append(dcm_path)
        index['by_name'].setdefault(dcm_path.name, dcm_path)
        try:
            dataset = pydicom.dcmread(str(dcm_path), stop_before_pixels=True)
        except Exception:
            continue
        for field_name in ('SOPInstanceUID', 'SeriesInstanceUID', 'StudyInstanceUID'):
            uid_value = getattr(dataset, field_name, None)
            if uid_value:
                index['by_uid'].setdefault(str(uid_value), dcm_path)
    return index

def resolve_image_path(xml_path, image_root, filename, raw_path, dicom_index):
    possible_paths = []
    if raw_path and raw_path.lower() != 'unknow':
        possible_paths.append(Path(raw_path))
        possible_paths.append(Path(image_root) / Path(raw_path).name)
    if filename and filename.lower() != 'unknow':
        possible_paths.append(Path(image_root) / filename)
        if filename in dicom_index['by_name']:
            return dicom_index['by_name'][filename]
    for path in possible_paths:
        if path.exists() and path.is_file():
            return path
    patient_id = normalize_patient_id(xml_path.parent.name)
    xml_uid = xml_path.stem
    uid_match = dicom_index['by_uid'].get(xml_uid)
    if uid_match is not None:
        if patient_id is None:
            return uid_match
        patient_files = dicom_index['by_patient'].get(patient_id, [])
        if uid_match in patient_files:
            return uid_match
    return None

def patient_id_from_path(image_root, image_path):
    try:
        relative = image_path.relative_to(image_root)
        if relative.parts:
            return normalize_patient_id(relative.parts[0])
    except ValueError:
        pass
    return normalize_patient_id(image_path.stem.split('_')[0])

def load_annotations(annotations_dir, image_root):
    annotations_dir = Path(annotations_dir)
    image_root = Path(image_root)
    xml_files = sorted(annotations_dir.rglob('*.xml'))
    if not xml_files:
        raise FileNotFoundError(f'No XML files found under {annotations_dir}')
    dicom_index = build_dicom_index(image_root)
    rows = []
    unresolved = []
    for xml_path in tqdm(xml_files, desc='Parsing XML annotations'):
        tree = ET.parse(xml_path)
        root = tree.getroot()
        filename = safe_text(root, 'filename')
        raw_path = safe_text(root, 'path')
        image_path = resolve_image_path(xml_path, image_root, filename, raw_path, dicom_index)
        if image_path is None:
            unresolved.append(str(xml_path))
            continue
        patient_id = patient_id_from_path(image_root, image_path)
        for obj in root.findall('object'):
            bbox = obj.find('bndbox')
            if bbox is None:
                continue
            try:
                xmin = int(float(safe_text(bbox, 'xmin') or 0))
                ymin = int(float(safe_text(bbox, 'ymin') or 0))
                xmax = int(float(safe_text(bbox, 'xmax') or 0))
                ymax = int(float(safe_text(bbox, 'ymax') or 0))
            except ValueError:
                continue
            if xmax <= xmin or ymax <= ymin:
                continue
            rows.append({
                'image_path': str(image_path),
                'patient_id': patient_id,
                'xml_path': str(xml_path),
                'object_name': (safe_text(obj, 'name') or 'tumor').lower(),
                'xmin': xmin,
                'ymin': ymin,
                'xmax': xmax,
                'ymax': ymax,
            })
    if not rows:
        raise RuntimeError('No usable annotation rows were parsed from the XML files.')
    frame = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    if unresolved:
        print(f'Warning: {len(unresolved)} XML files could not be resolved to a local DICOM slice.')
        print('First unresolved XML:', unresolved[0])
    return frame


## CT Helpers

These functions load a DICOM slice, apply windowing, and make crops.


In [ ]:
def load_dicom_image(dicom_path):
    dataset = pydicom.dcmread(dicom_path)
    pixels = dataset.pixel_array.astype(np.float32)
    slope = float(getattr(dataset, 'RescaleSlope', 1.0))
    intercept = float(getattr(dataset, 'RescaleIntercept', 0.0))
    return pixels * slope + intercept

def apply_window(image, center=40.0, width=350.0):
    low = center - width / 2.0
    high = center + width / 2.0
    image = np.clip(image, low, high)
    image = (image - low) / max(width, 1e-6)
    return image.astype(np.float32)

def clip_box(box, image_shape):
    height, width = image_shape
    xmin, ymin, xmax, ymax = box
    xmin = max(0, min(xmin, width - 1))
    ymin = max(0, min(ymin, height - 1))
    xmax = max(xmin + 1, min(xmax, width))
    ymax = max(ymin + 1, min(ymax, height))
    return xmin, ymin, xmax, ymax

def add_padding(box, image_shape, padding=0.25):
    xmin, ymin, xmax, ymax = clip_box(box, image_shape)
    box_width = xmax - xmin
    box_height = ymax - ymin
    pad_x = int(box_width * padding)
    pad_y = int(box_height * padding)
    return clip_box((xmin - pad_x, ymin - pad_y, xmax + pad_x, ymax + pad_y), image_shape)

def crop_box(image, box):
    xmin, ymin, xmax, ymax = clip_box(box, image.shape)
    crop = image[ymin:ymax, xmin:xmax]
    if crop.size == 0:
        return np.zeros((16, 16), dtype=np.float32)
    return crop

def compute_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_width = max(0, inter_x2 - inter_x1)
    inter_height = max(0, inter_y2 - inter_y1)
    inter_area = inter_width * inter_height
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter_area
    if union == 0:
        return 0.0
    return inter_area / union

def sample_background_box(image_shape, tumor_box, padding, rng):
    height, width = image_shape
    tumor_box = add_padding(tumor_box, image_shape, padding)
    tumor_width = max(16, tumor_box[2] - tumor_box[0])
    tumor_height = max(16, tumor_box[3] - tumor_box[1])
    tumor_width = min(tumor_width, width)
    tumor_height = min(tumor_height, height)
    max_x = max(0, width - tumor_width)
    max_y = max(0, height - tumor_height)
    for _ in range(50):
        xmin = int(rng.integers(0, max_x + 1)) if max_x > 0 else 0
        ymin = int(rng.integers(0, max_y + 1)) if max_y > 0 else 0
        candidate = (xmin, ymin, xmin + tumor_width, ymin + tumor_height)
        if compute_iou(candidate, tumor_box) < 0.05:
            return candidate
    return (0, 0, tumor_width, tumor_height)

def resize_crop(image, image_size=128):
    image = image[..., np.newaxis]
    image = tf.image.resize(image, size=(image_size, image_size)).numpy()
    return image.astype(np.float32)


## Dataset Creation

This section creates NumPy arrays for Keras.


In [ ]:
def build_sample_list(annotations, negatives_per_positive=1, seed=42):
    rng = np.random.default_rng(seed)
    samples = []
    for row in annotations.itertuples(index=False):
        tumor_box = (int(row.xmin), int(row.ymin), int(row.xmax), int(row.ymax))
        samples.append({'image_path': str(row.image_path), 'xmin': tumor_box[0], 'ymin': tumor_box[1], 'xmax': tumor_box[2], 'ymax': tumor_box[3], 'label': 1, 'seed': int(rng.integers(0, 2**31 - 1))})
        for _ in range(negatives_per_positive):
            samples.append({'image_path': str(row.image_path), 'xmin': tumor_box[0], 'ymin': tumor_box[1], 'xmax': tumor_box[2], 'ymax': tumor_box[3], 'label': 0, 'seed': int(rng.integers(0, 2**31 - 1))})
    return samples

def augment_image(image, rng):
    if rng.random() < 0.5:
        image = np.flip(image, axis=1)
    if rng.random() < 0.5:
        image = np.flip(image, axis=0)
    if rng.random() < 0.2:
        noise = rng.normal(0.0, 0.01, size=image.shape)
        image = np.clip(image + noise, 0.0, 1.0)
    return image.astype(np.float32)

def create_numpy_dataset(annotations, image_size=128, negatives_per_positive=1, window_center=40.0, window_width=350.0, padding=0.25, augment=False, seed=42):
    samples = build_sample_list(annotations, negatives_per_positive=negatives_per_positive, seed=seed)
    images = []
    labels = []
    for sample in samples:
        image = load_dicom_image(str(sample['image_path']))
        image = apply_window(image, center=window_center, width=window_width)
        tumor_box = (int(sample['xmin']), int(sample['ymin']), int(sample['xmax']), int(sample['ymax']))
        if int(sample['label']) == 1:
            crop_area = add_padding(tumor_box, image.shape, padding)
        else:
            rng = np.random.default_rng(int(sample['seed']))
            crop_area = sample_background_box(image.shape, tumor_box, padding, rng)
        crop = crop_box(image, crop_area)
        crop = resize_crop(crop, image_size)
        if augment:
            rng = np.random.default_rng(int(sample['seed']))
            crop = augment_image(crop, rng)
        images.append(crop)
        labels.append(float(sample['label']))
    x = np.asarray(images, dtype=np.float32)
    y = np.asarray(labels, dtype=np.float32)
    return x, y


## Model

A very small CNN built with Keras.


In [ ]:
def build_model(image_size=128):
    model = keras.Sequential([
        keras.layers.Input(shape=(image_size, image_size, 1)),
        keras.layers.Conv2D(16, kernel_size=3, activation='relu', padding='same'),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(32, kernel_size=3, activation='relu', padding='same'),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, kernel_size=3, activation='relu', padding='same'),
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(1, activation='sigmoid'),
    ])
    return model


## Training Functions

This cell handles splitting, training, and saving the best model.


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def split_by_patient(annotations, seed=42, val_size=0.15, test_size=0.15):
    patient_ids = list(annotations['patient_id'].dropna().unique())
    if len(patient_ids) < 3:
        raise ValueError('Need at least 3 patients to make train/val/test splits.')
    rng = random.Random(seed)
    rng.shuffle(patient_ids)
    test_count = max(1, int(len(patient_ids) * test_size))
    val_count = max(1, int(len(patient_ids) * val_size))
    if test_count + val_count >= len(patient_ids):
        test_count = 1
        val_count = 1
    test_patients = set(patient_ids[:test_count])
    val_patients = set(patient_ids[test_count:test_count + val_count])
    train_patients = set(patient_ids[test_count + val_count:])
    train_df = annotations[annotations['patient_id'].isin(train_patients)].reset_index(drop=True)
    val_df = annotations[annotations['patient_id'].isin(val_patients)].reset_index(drop=True)
    test_df = annotations[annotations['patient_id'].isin(test_patients)].reset_index(drop=True)
    return train_df, val_df, test_df

def train_model(image_root, annotations_dir, output_dir, epochs=10, batch_size=16, lr=1e-3, image_size=128, negatives_per_positive=1, window_center=40.0, window_width=350.0, padding=0.25, val_size=0.15, test_size=0.15, seed=42):
    set_seed(seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    annotations = load_annotations(annotations_dir, image_root)
    annotations.to_csv(output_dir / 'annotations_resolved.csv', index=False)
    train_df, val_df, test_df = split_by_patient(annotations, seed=seed, val_size=val_size, test_size=test_size)
    train_df.to_csv(output_dir / 'train_annotations.csv', index=False)
    val_df.to_csv(output_dir / 'val_annotations.csv', index=False)
    test_df.to_csv(output_dir / 'test_annotations.csv', index=False)
    x_train, y_train = create_numpy_dataset(train_df, image_size=image_size, negatives_per_positive=negatives_per_positive, window_center=window_center, window_width=window_width, padding=padding, augment=True, seed=seed)
    x_val, y_val = create_numpy_dataset(val_df, image_size=image_size, negatives_per_positive=negatives_per_positive, window_center=window_center, window_width=window_width, padding=padding, augment=False, seed=seed + 1)
    x_test, y_test = create_numpy_dataset(test_df, image_size=image_size, negatives_per_positive=negatives_per_positive, window_center=window_center, window_width=window_width, padding=padding, augment=False, seed=seed + 2)
    model = build_model(image_size=image_size)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='binary_crossentropy', metrics=['accuracy'])
    checkpoint_path = output_dir / 'best.keras'
    callbacks = [tf.keras.callbacks.ModelCheckpoint(filepath=str(checkpoint_path), monitor='val_accuracy', mode='max', save_best_only=True)]
    history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=epochs, batch_size=batch_size, callbacks=callbacks, verbose=1)
    best_model = tf.keras.models.load_model(checkpoint_path)
    test_loss, test_accuracy = best_model.evaluate(x_test, y_test, verbose=0)
    metadata = {'image_size': image_size, 'window_center': window_center, 'window_width': window_width, 'padding': padding}
    with (output_dir / 'metrics.json').open('w', encoding='utf-8') as file:
        json.dump({'history': history.history, 'test_metrics': {'loss': float(test_loss), 'accuracy': float(test_accuracy)}, 'train_annotations': len(train_df), 'val_annotations': len(val_df), 'test_annotations': len(test_df), **metadata}, file, indent=2)
    with (output_dir / 'best_info.json').open('w', encoding='utf-8') as file:
        json.dump(metadata, file, indent=2)
    print('Saved model to', checkpoint_path)
    print({'test_loss': float(test_loss), 'test_accuracy': float(test_accuracy)})
    return best_model, history.history


## Set Paths

Edit these paths before running the training cell.


In [ ]:
IMAGE_ROOT = Path(r'C:/Users/Usuario/Documents/Lung Cancer/Lung-PET-CT-Dx-Annotations-XML-Files-rev12222020/lung_pet_ct_dx')
ANNOTATIONS_DIR = Path(r'C:/Users/Usuario/Documents/Lung Cancer/Lung-PET-CT-Dx-Annotations-XML-Files-rev12222020/Annotation')
OUTPUT_DIR = Path(r'C:/Users/Usuario/Documents/Lung Cancer/Lung-PET-CT-Dx-Annotations-XML-Files-rev12222020/notebook_output')

IMAGE_SIZE = 128
NEGATIVES_PER_POSITIVE = 1
WINDOW_CENTER = 40.0
WINDOW_WIDTH = 350.0
PADDING = 0.25
EPOCHS = 10
BATCH_SIZE = 16
SEED = 42


## Train

Run this cell after editing the paths above.


In [ ]:
# Uncomment to train the model.
# model, history = train_model(
#     image_root=IMAGE_ROOT,
#     annotations_dir=ANNOTATIONS_DIR,
#     output_dir=OUTPUT_DIR,
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     lr=LEARNING_RATE,
#     image_size=IMAGE_SIZE,
#     negatives_per_positive=NEGATIVES_PER_POSITIVE,
#     window_center=WINDOW_CENTER,
#     window_width=WINDOW_WIDTH,
#     padding=PADDING,
#     seed=SEED,
# )


## Inference

This function predicts the probability for one crop.


In [ ]:
def predict_crop(checkpoint_path, dicom_path, xmin, ymin, xmax, ymax, image_size=None, window_center=None, window_width=None, padding=None):
    checkpoint_path = Path(checkpoint_path)
    model = tf.keras.models.load_model(checkpoint_path)
    metadata_path = checkpoint_path.with_name(checkpoint_path.stem + '_info.json')
    metadata = {}
    if metadata_path.exists():
        with metadata_path.open('r', encoding='utf-8') as file:
            metadata = json.load(file)
    image_size = image_size or metadata.get('image_size', 128)
    window_center = window_center or metadata.get('window_center', 40.0)
    window_width = window_width or metadata.get('window_width', 350.0)
    padding = padding or metadata.get('padding', 0.25)
    image = load_dicom_image(dicom_path)
    image = apply_window(image, center=window_center, width=window_width)
    box = add_padding((xmin, ymin, xmax, ymax), image.shape, padding)
    crop = crop_box(image, box)
    tensor = resize_crop(crop, image_size)[np.newaxis, ...]
    probability = float(model.predict(tensor, verbose=0)[0][0])
    print(f'tumor_probability={probability:.4f}')
    return probability


In [ ]:
# Example inference call after training:
# probability = predict_crop(
#     checkpoint_path=OUTPUT_DIR / 'best.keras',
#     dicom_path=IMAGE_ROOT / 'example.dcm',
#     xmin=120,
#     ymin=140,
#     xmax=220,
#     ymax=240,
# )
